# WhatsApp Chat Analysis

Standalone notebook version. Run the cells from top to bottom.

In [ ]:
from pathlib import Path
from collections import Counter
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from wordcloud import WordCloud
from urlextract import URLExtract
import emoji

# Import the Hinglish stop-word list from the project file.
STOP_WORDS_FILE = Path('stop_hinglish.txt')
if not STOP_WORDS_FILE.exists():
    raise FileNotFoundError(f'Missing required file: {STOP_WORDS_FILE.resolve()}')
STOP_WORDS = {line.strip().casefold() for line in STOP_WORDS_FILE.read_text(encoding='utf-8').splitlines() if line.strip()}

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## Embedded analysis functions

The next cell contains all parsing, statistics, timeline, activity, word, and emoji functions used by this notebook.

In [ ]:
def parse_chat(data):
    pattern = r'\d{1,2}/\d{1,2}/\d{2,4},\s\d{1,2}:\d{2}(?:\s?[APap][Mm])?\s-\s'
    messages = re.split(pattern, data)[1:]
    dates = re.findall(pattern, data)
    frame = pd.DataFrame({'raw_message': messages, 'message_date': dates})
    frame['message_date'] = pd.to_datetime(frame['message_date'].str.replace(' - ', '', regex=False), format='mixed')
    users, texts = [], []
    for message in frame['raw_message']:
        parts = re.split(r'([\w\W]+?):\s', message)
        if parts[1:]:
            users.append(parts[1]); texts.append(parts[2])
        else:
            users.append('group_notification'); texts.append(parts[0])
    frame['user'] = users
    frame['text'] = texts
    frame['only_date'] = frame['message_date'].dt.date
    frame['year'] = frame['message_date'].dt.year
    frame['month_num'] = frame['message_date'].dt.month
    frame['month'] = frame['message_date'].dt.month_name()
    frame['day_name'] = frame['message_date'].dt.day_name()
    frame['hour'] = frame['message_date'].dt.hour
    frame['period'] = frame['hour'].apply(lambda hour: f'{int(hour):02d}-{(int(hour) + 1) % 24:02d}' if pd.notna(hour) else 'Unknown')
    frame['minute'] = frame['message_date'].dt.minute
    frame.drop(columns=['raw_message', 'message_date'], inplace=True)
    return frame[(frame['user'] != 'group_notification') & frame['text'].notna()]

url_finder = URLExtract()

def get_stats(selected_user, frame):
    if selected_user != 'Overall': frame = frame[frame['user'] == selected_user]
    words = [word for text in frame['text'] for word in text.split()]
    media = frame[frame['text'] == '<Media omitted>\n'].shape[0]
    links = [url for text in frame['text'] for url in url_finder.find_urls(text)]
    return frame.shape[0], len(words), media, len(links)

def daily_counts(selected_user, frame):
    if selected_user != 'Overall': frame = frame[frame['user'] == selected_user]
    result = frame.groupby('only_date').count()['text'].reset_index()
    result['text'] = result['text'].astype(int)
    return result

def monthly_counts(selected_user, frame):
    if selected_user != 'Overall': frame = frame[frame['user'] == selected_user]
    result = frame.groupby(['year', 'month_num', 'month']).count()['text'].reset_index()
    result['year'] = result['year'].astype(int)
    result['text'] = result['text'].astype(int)
    result['period_label'] = result['month'] + '-' + result['year'].astype(str)
    return result

def day_counts(selected_user, frame):
    if selected_user != 'Overall': frame = frame[frame['user'] == selected_user]
    return frame['day_name'].value_counts()

def month_counts(selected_user, frame):
    if selected_user != 'Overall': frame = frame[frame['user'] == selected_user]
    return frame['month'].value_counts()

def time_heatmap(selected_user, frame):
    if selected_user != 'Overall': frame = frame[frame['user'] == selected_user]
    periods = [f'{hour:02d}-{(hour + 1) % 24:02d}' for hour in range(24)]
    days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    return frame.pivot_table(index='day_name', columns='period', values='text', aggfunc='count', fill_value=0).reindex(index=days, columns=periods, fill_value=0).astype(int)

def top_users(frame):
    counts = frame['user'].value_counts().head()
    percentages = round((frame['user'].value_counts() / frame.shape[0]) * 100, 2).reset_index()
    percentages.columns = ['User', 'Percentage']
    percentages.index = range(1, len(percentages) + 1)
    return counts, percentages

def clean_words(selected_user, frame):
    if selected_user != 'Overall': frame = frame[frame['user'] == selected_user]
    return [word for text in frame[frame['text'].str.strip() != '<Media omitted>']['text'] for word in text.split() if word.casefold() not in STOP_WORDS]

def make_wordcloud(selected_user, frame):
    return WordCloud(width=500, height=500, min_font_size=10, background_color='white').generate(' '.join(clean_words(selected_user, frame)))

def common_words(selected_user, frame):
    return pd.DataFrame(Counter(clean_words(selected_user, frame)).most_common(20))

def emoji_counts(selected_user, frame):
    if selected_user != 'Overall': frame = frame[frame['user'] == selected_user]
    found = [character for text in frame['text'] for character in text if character in emoji.EMOJI_DATA]
    result = pd.DataFrame(Counter(found).most_common(len(Counter(found))), columns=['Emoji', 'Count'])
    result.index = range(1, len(result) + 1)
    return result

In [ ]:
chat_path = Path('chat.txt')
data = chat_path.read_text(encoding='utf-8')
df = parse_chat(data)
user_list = sorted(df['user'].unique().tolist())
user_list.insert(0, 'Overall')
selected_user = 'Overall'
print(f'Loaded {chat_path.name}')
print(f'Messages: {len(df):,}')
print(user_list)

## Statistics

In [ ]:
num_messages, word_count, media_count, link_count = get_stats(selected_user, df)
pd.DataFrame({
    'Metric': ['Messages', 'Words', 'Media', 'Links'],
    'Value': [num_messages, word_count, media_count, link_count]
}, index=range(1, 5))

## Monthly Timeline

In [ ]:
timeline = monthly_counts(selected_user, df)
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(timeline['period_label'], timeline['text'], color='green', marker='o')
ax.set_title('Messages Over Months')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Messages')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Daily Timeline

In [ ]:
daily = daily_counts(selected_user, df)
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily['only_date'], daily['text'], color='black')
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%m-%Y'))
ax.set_title('Messages by Date')
ax.set_xlabel('Date')
ax.set_ylabel('Number of Messages')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Activity Charts

In [ ]:
busy_day = day_counts(selected_user, df)
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(busy_day.index, busy_day.values, color='purple')
ax.set_title('Messages by Day of Week')
ax.set_xlabel('Day')
ax.set_ylabel('Number of Messages')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
busy_month = month_counts(selected_user, df)
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(busy_month.index, busy_month.values, color='orange')
ax.set_title('Messages by Month')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Messages')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Weekly Activity Map

In [ ]:
heatmap_data = time_heatmap(selected_user, df)
fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(heatmap_data, ax=ax, cmap='YlGnBu', annot=True, fmt='.0f', linewidths=0.5, linecolor='white')
ax.set_title('Messages by Day and Time')
ax.set_xlabel('Time Period')
ax.set_ylabel('Day')
plt.tight_layout()
plt.show()

## Participants

In [ ]:
top_user_counts, participant_table = top_users(df)
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(top_user_counts.index, top_user_counts.values, color='red')
ax.set_title('Top Participants by Message Count')
ax.set_xlabel('Users')
ax.set_ylabel('Number of Messages')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
participant_table.style.format({'Percentage': '{:,.2f}%'})

## Wordcloud

In [ ]:
cloud = make_wordcloud(selected_user, df)
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(cloud)
ax.axis('off')
plt.show()

## Most Common Words

In [ ]:
common = common_words(selected_user, df)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(common[0], common[1])
ax.set_title('Most Frequently Used Words')
ax.set_xlabel('Number of Occurrences')
ax.set_ylabel('Words')
plt.tight_layout()
plt.show()

## Emojis

In [ ]:
emoji_table = emoji_counts(selected_user, df)
emoji_table.style.format({'Count': '{:,}'})

In [ ]:
emoji_data = emoji_table.head(5)
fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(emoji_data['Count'], labels=emoji_data['Emoji'], autopct='%1.2f%%', textprops={'fontname': 'Segoe UI Emoji', 'fontsize': 16})
ax.set_title('Emoji Usage Distribution')
plt.show()